EX2: Train the GPT on your own dataset of choice! What other data could be fun to blabber on about? (A fun advanced suggestion if you like: train a GPT to do addition of two numbers, i.e. a+b=c. You may find it helpful to predict the digits of c in reverse order, as the typical addition algorithm (that you're hoping it learns) would proceed right to left too. You may want to modify the data loader to simply serve random problems and skip the generation of train.bin, val.bin. You may want to mask out the loss at the input positions of a+b that just specify the problem using y=-1 in the targets (see CrossEntropyLoss ignore_index). Does your Transformer learn to add? Once you have this, swole doge project: build a calculator clone in GPT, for all of +-*/. Not an easy problem. You may need Chain of Thought traces.)

In [1]:
import random
import torch
import torch.nn as nn
from torch.nn import functional as F

In [2]:
# hyperparameters
batch_size = 6
block_size = 10
n_embd = 32

n_head= 4
head_dim = 8

END = '@' # 结束符
PAD = '_' # 占位符
# ===============

In [3]:
# mapping
chars = "".join(str(i) for i in range(0, 10))
stoi = { s:i for i,s in enumerate(chars)}
stoi['+'] = 10
stoi['='] = 11
stoi[END] = 12 # 结束符号
stoi[PAD] = 13 # pad
itos = { i:s for s, i in stoi.items()}

vocab_size = len(itos)

In [4]:
# 训练集与测试集
random.seed(22)

pairlist = []
for a in range(10, 100):
    for b in range(a,100):
        pairlist.append((a,b))
random.shuffle(pairlist)
n = int(len(pairlist) * 0.9)
train_data = pairlist[:n]
val_data = pairlist[n:]


In [5]:
# 训练集测试集相交检验
set_test = set(train_data)
set_val = set(val_data)
set_test.isdisjoint(set_val)

True

In [6]:
def encode(s, is_y = False):
    out = torch.tensor([stoi[c] for c in s])
    if (is_y):
        befor_eq = False
        for i in range(out.shape[0]):
            if not befor_eq:
                befor_eq = befor_eq or out[i] == stoi['='] 
                out[i] = -1
            if out[i] == stoi[PAD]:
                out[i] = -1
            
    return out

def get_batch(split):
    data = train_data if split == 'train' else val_data
    # 一个pair交互顺序能生成两条式子
    expressions = []
    for i in range(int(batch_size/2)):
        pair = random.choice(data)
        a = pair[0]; b= pair[1]; c = a + b
        expressions.append(f"{a}+{b}={str(c)[::-1]}@")
        expressions.append(f"{b}+{a}={str(c)[::-1]}@")
    x = torch.stack([encode(exp.ljust(block_size, PAD)) for exp in expressions])
    y = torch.stack([encode(exp[1:].ljust(block_size, PAD), True) for exp in expressions])
    return x,y

In [7]:
# 输入
random.seed(22)
idx,target = get_batch('train')

In [8]:
# embeding
token_emb_table = nn.Embedding(vocab_size, n_embd) 
pos_emb_table = nn.Embedding(block_size, n_embd)

B,T = idx.shape
pos_emb = pos_emb_table(torch.arange(T))
tok_emb = token_emb_table(idx)

x = pos_emb + tok_emb
x.shape

torch.Size([6, 10, 32])

In [9]:
# layerNorm 1
layerNormal_1 =  nn.LayerNorm(n_embd)
ln_1 = layerNormal_1(x)
ln_1.shape

torch.Size([6, 10, 32])

In [10]:
# multi head attention
class MultiHead(nn.Module):
    def __init__(self, num_head, head_dim):
        super().__init__()
        self.num_head = num_head
        self.head_dim = head_dim
        self.q_proj = nn.Linear(n_embd, head_dim * num_head, bias=False)
        self.k_proj = nn.Linear(n_embd, head_dim * num_head, bias=False)
        self.v_proj = nn.Linear(n_embd, head_dim * num_head, bias=False)
        self.linear = nn.Linear(head_dim * num_head, n_embd, bias=False)

    # BTC => BHTD
    def tran(self, batch_size, time_size, input : torch.Tensor):
        out = input.view(batch_size, time_size, self.num_head, self.head_dim) # BTHD
        out = out.transpose(dim0=1, dim1=2)
        return out

    def forward(self, x):
        B, T, C = x.shape
        Q = self.tran(B, T, self.q_proj(x))
        K = self.tran(B, T, self.k_proj(x))
        V = self.tran(B, T, self.v_proj(x))

        S = Q @ K.transpose(-1,-2) * (self.head_dim **(-0.5))
        mask = torch.tril(torch.ones(T, T))
        S = S.masked_fill(mask==0, float('-inf'))
        A = F.softmax(S, dim=-1)
        O = A @ V

        O = O.transpose(1,2) # 先交换维度
        O = O.reshape(B, T, self.num_head * self.head_dim)
        out = self.linear(O)
        return out


In [11]:
multi_head = MultiHead(n_head, head_dim)
attention = multi_head(ln_1)
attention.shape

torch.Size([6, 10, 32])

In [12]:
# 残差连接
u = x + attention
u.shape

torch.Size([6, 10, 32])

In [13]:
class Feed_Froward(nn.Module):
    def __init__(self):
        super().__init__()
        self.fnn = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.GELU(),
            nn.Linear(4 * n_embd, n_embd)
        )

    def forward(self, x):
        return self.fnn(x)


In [14]:
fnn = Feed_Froward()
fnn_out = fnn(u)
fnn_out.shape

torch.Size([6, 10, 32])

In [15]:
z = u + fnn_out

In [48]:
final_ln = nn.LayerNorm(n_embd)
final_linear = nn.Linear(n_embd, vocab_size)
z1 =  final_ln(z)
logits = final_linear(z1)
B,T,S = logits.shape
logits = logits.view(B*T, -1)
target = target.view(B*T,)

In [49]:
target.shape

torch.Size([60])

In [50]:
loss = F.cross_entropy(logits, target, ignore_index=-1)

In [43]:
loss

tensor(2.8574, grad_fn=<NllLossBackward0>)